# 🏗️ PyTorch 한영 번역 — Seq2Seq에서 Transformer까지 (건설 문장)

**건축공학딥러닝프로그래밍 · PyTorch 한영 번역 Seq2Seq→Transformer · 실습 노트북 (실습본 · 빈칸 채우기)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_DL-AE/blob/main/PyTorch_KoEn_Seq2Seq_Transformer/koen_steps_practice.ipynb) ← **실습본**(빈칸 채우기) &nbsp;&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_DL-AE/blob/main/PyTorch_KoEn_Seq2Seq_Transformer/koen_steps_complete.ipynb) ← **완성본**(정답·실행결과 포함)

> "어제 작업자가 3층 기둥의 균열을 보수했다" → **"the worker repaired the crack in the column on the 3rd floor yesterday"**
> 한국어는 **누가 – 무엇을 – 했다** 순서이고, 영어는 **누가 – 했다 – 무엇을** 순서입니다. 낱말을 바꾸는 것만으로는 번역이 안 되고, **순서까지 다시 짜야** 합니다.
> 이 노트북은 짧은 건설 문장 수천 쌍을 **직접 만들고**, 같은 데이터로 **Seq2Seq(GRU) → 어텐션 → Transformer** 를 차례로 **처음부터 학습**해 비교합니다. 미리 학습한 모델은 쓰지 않습니다.

## 📌 강의 중점
- 한국어 글자 · 영어 단어를 **토큰 → 번호(어휘 사전) → 벡터(임베딩)** 로 바꾸기 · 특수 토큰 `<pad>` `<sos>` `<eos>`
- **teacher forcing**: 디코더 입력과 정답이 한 칸 어긋난 텐서를 직접 확인
- 학습 루프 4단계와 **loss 곡선**(학습 · 검증) 읽기
- 한 단어씩 이어 붙이는 **탐욕 디코딩**, 번역 점수 **BLEU** 와 **완전 일치**
- (5–7장) 컨텍스트 벡터 하나의 **병목** → **어텐션** → **Transformer**(셀프 어텐션 · 위치 · 마스크) 비교

## 🎯 학습 목표
1. 용어 사전과 문형으로 한-영 문장 쌍을 만들고, `[배치, 길이]` 번호 텐서로 바꿀 수 있다.
2. 디코더 학습이 "정답 앞부분을 보고 **다음 단어를 분류**"하는 것임을 teacher forcing 텐서로 설명할 수 있다.
3. 학습 루프를 직접 돌리고, 학습 · 검증 loss 곡선으로 학습이 잘 되는지 판단할 수 있다.
4. 탐욕 디코딩으로 번역하고 BLEU · 완전 일치로 채점해, 어디서 틀리는지 찾을 수 있다.

## 📋 이 노트북 사용법

| 표시 | 뜻 |
|:-:|---|
| ▶ | 셀 실행 — 셀을 클릭하고 `Shift + Enter` |
| 🎬 EP*n* | 이 부분을 설명하는 강의 영상 에피소드 (e-campus) |
| ⏸ | 영상을 멈추고 직접 실행해 보는 지점 |
| ✅ / ❌ | 자가검증 결과 — ❌ 가 나오면 힌트를 읽고 고쳐 보세요 |
| 🔸 | 실험 결과 안내 (오류가 아님) |
| ✏️ | 빈칸 `____` 을 채워야 하는 줄 (주석 끝의 힌트 참고) |

**주석(`#` 뒤의 글) 읽는 법** — 실행에는 영향이 없는 설명입니다.
- `①②③`, `1️⃣2️⃣3️⃣4️⃣` 순서(단계) · `💡` 이론 연결 · `📐` 데이터 모양(shape) · `⚠️` 흔한 실수 · `✅` 기대 결과 · `⏱` 걸린 시간

**Colab에서 처음 열었다면**
1. 메뉴 **파일 → Drive에 사본 저장** 을 먼저 누르세요. (사본에서 작업해야 내 Drive에 저장됩니다)
2. 데이터는 노트북 안에서 **직접 만듭니다** (내려받기 없음). CPU 그대로 두어도 되고, **GPU 런타임**(메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**)이면 더 빠릅니다.
3. 위에서부터 **차례대로** 실행하세요. 런타임이 초기화되어 변수가 사라지면 처음부터 다시 실행하면 됩니다.
4. ✏️ 표시된 줄의 `____` 을 채운 뒤 실행하세요. **'모두 실행'은 첫 빈칸에서 멈춥니다** — 하나씩 채우며 내려가세요.

**🎬 영상과 함께 보는 법 (반복 학습 추천 순서)**
1회차 영상 전체 보기 → 2회차 실습본을 열고 ⏸ 마다 멈추며 따라 치기 → 3회차 영상 없이 빈칸 채우기 → 모든 ✅ 통과

---
## 0. 준비 — 라이브러리 불러오기  🎬 EP1

이 셀에서 할 일: PyTorch와 그래프 도구를 불러오고, 계산할 장치(CPU 또는 GPU)를 정합니다.

> 💡 **새 파이썬 문법 — random**
>
> `random` 은 파이썬 기본 난수 도구입니다. `rng = random.Random(0)` 처럼 **씨앗(seed)** 을 주면 매번 같은 순서의 난수가 나옵니다.
> `rng.choice(목록)` 은 하나 고르기, `rng.randint(1, 15)` 는 1 ~ 15 사이 정수, `rng.randrange(3)` 은 0 · 1 · 2 중 하나입니다.

In [ ]:
import math                                    # 로그 · 지수 (BLEU 계산)
import random                                  # 파이썬 기본 난수 (문장 만들기)
import time                                    # 걸린 시간 재기 (⏱)
from collections import Counter                # 개수 세기 (BLEU 계산)
import torch                                   # PyTorch: 딥러닝 도구 상자
from torch import nn                           # nn = 신경망 부품(층) 모음
import matplotlib.pyplot as plt                # 그래프 도구 (별명 plt)

torch.set_printoptions(precision=4, sci_mode=False)      # 소수 4자리, 지수 표기 끄기
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU가 있으면 GPU, 없으면 CPU
print("PyTorch 버전:", torch.__version__)                # Colab에는 이미 설치됨
print("계산 장치:", device)                              # cuda = GPU, cpu = CPU

### 자가검증 도우미 `check()` · `check_range()` 준비
아래 셀은 결과가 맞는지 ✅ / ❌ 로 알려 주는 **도우미 함수**입니다. 지난 실습과 같습니다.
- `check()` : 기대값과 허용오차(`tol`) 이내로 같으면 ✅ (`tol=0` 이면 완전히 같아야 ✅)
- `check_range()` : 정확도처럼 **범위 안에 들어오면** ✅ (컴퓨터 · GPU 마다 조금씩 달라지는 값에 씁니다)

In [ ]:
def check(name, value, expected, tol=0.01):  # 자가검증 (이름, 결과, 기대값, 허용오차)
    """value 가 expected 와 tol 이내로 같으면 ✅, 다르면 ❌ 를 보여 줍니다."""
    value = torch.as_tensor(value, dtype=torch.float32)        # 숫자·리스트 → 텐서
    expected = torch.as_tensor(expected, dtype=torch.float32)  # 기대값도 텐서로
    if value.shape != expected.shape:                          # ⚠️ 모양부터 다르면
        print(f"❌ {name}: 모양 {tuple(value.shape)} ≠ 기대 {tuple(expected.shape)}")
        return                                                 # 여기서 끝
    gap = (value - expected).abs().max().item()                # 가장 큰 차이
    mark = "✅" if gap <= tol else "❌"                         # 허용오차 안이면 통과
    tol_s = f"{tol:.6f}".rstrip("0").rstrip(".")               # 0.000001 처럼 보기 좋게
    print(f"{mark} {name}: 최대 차이 {gap:.6f} (허용 {tol_s})")  # 결과 출력

def check_range(name, value, lo, hi):        # 자가검증 (이름, 결과, 최솟값, 최댓값)
    """value 가 lo ~ hi 범위 안이면 ✅, 벗어나면 ❌ 를 보여 줍니다."""
    value = float(value)                                       # 텐서·숫자 → 파이썬 실수
    mark = "✅" if lo <= value <= hi else "❌"                  # 범위 안이면 통과
    print(f"{mark} {name}: {value:.4f} (기대 범위 {lo} ~ {hi})")  # 결과 출력

---
## 1. 번역 문제와 데이터 — 건설 문장 만들기  🎬 EP1

번역기는 **같은 뜻의 한국어 · 영어 문장 쌍**을 많이 보고 배웁니다. 이번에는 문장 쌍을 내려받지 않고 **용어 사전과 문형으로 직접 만듭니다.**

| | 사람 | 시간 | 장소 · 대상 | 동작 |
|---|---|---|---|---|
| 한국어 | 작업자가 | 어제 | 3층 기둥의 균열을 | 보수했다 |
| 영어 | the worker | yesterday | the crack in the column on the 3rd floor | repaired |

영어 문장은 **사람 → 동작 → 대상 → 시간** 순서로 놓입니다. 한국어의 맨 끝에 있던 동작이 영어에서는 두 번째로 옵니다. 모델은 이 **순서 바꾸기**까지 배워야 합니다.

### 1-1. 용어 사전 — 사람 · 부재 · 하자
각 줄은 `(한국어, 영어)` 짝입니다. 부재마다 **함께 쓰는 하자**를 정해 두어, "창호의 처짐"처럼 어색한 조합은 만들지 않습니다.

In [ ]:
SUBJ = [("작업자", "the worker"), ("감리자", "the supervisor"),           # 사람 (주어)
        ("현장소장", "the site manager"), ("안전관리자", "the safety manager"),  # 사람
        ("점검팀", "the inspection team"), ("협력업체", "the subcontractor"),  # 사람
        ("구조기술사", "the structural engineer")]                       # 7명
MEMBER = [("기둥", "column"), ("보", "beam"), ("슬래브", "slab"),        # 부재
          ("벽체", "wall"), ("계단", "staircase"), ("발코니", "balcony"),  # 부재
          ("천장", "ceiling"), ("창호", "window")]                       # 8개
DEFECT = {"균열": "crack", "누수": "leak", "박락": "spalling",          # 하자 (한 → 영)
          "처짐": "deflection", "부식": "corrosion", "들뜸": "delamination",  # 하자
          "백화": "efflorescence", "변형": "deformation"}                # 8개
FITS = {"기둥": ["균열", "박락", "부식", "변형"],      # 부재마다 함께 쓰는 하자
        "보": ["균열", "처짐", "박락", "변형"],        # 보
        "슬래브": ["균열", "처짐", "누수", "박락"],    # 슬래브
        "벽체": ["균열", "누수", "백화", "들뜸"],      # 벽체
        "계단": ["균열", "박락", "들뜸"],              # 계단
        "발코니": ["누수", "균열", "백화", "부식"],    # 발코니
        "천장": ["누수", "처짐", "들뜸"],              # 천장
        "창호": ["누수", "변형", "부식"]}              # 창호
print("사람", len(SUBJ), "· 부재", len(MEMBER), "· 하자", len(DEFECT))  # 용어 수 확인

### 1-2. 동작과 시제
동작마다 영어의 **과거 · 현재 · 미래** 형태를 함께 적습니다. 시제는 한국어 끝(`했다 · 한다 · 할 것이다`)과 시간 표현(`어제 · 매일 · 내일` …)이 함께 정합니다.

In [ ]:
VERB = [("보수", ("repaired", "repairs", "will repair")),  # (한국어, (과거, 현재, 미래))
        ("점검", ("inspected", "inspects", "will inspect")),      # 점검
        ("발견", ("found", "finds", "will find")),                # 발견
        ("기록", ("recorded", "records", "will record")),         # 기록
        ("측정", ("measured", "measures", "will measure")),       # 측정
        ("확인", ("checked", "checks", "will check")),            # 확인
        ("촬영", ("photographed", "photographs", "will photograph"))]  # 촬영
REPORT = ("reported", "reports", "will report")             # 긴 문장의 '보고'
ADV = {0: [("어제", "yesterday"), ("지난주에", "last week")],  # 0 = 과거
       1: [("매일", "every day"), ("매주", "every week")],    # 1 = 현재
       2: [("내일", "tomorrow"), ("다음 주에", "next week")]}  # 2 = 미래
KO_END = {0: "했다", 1: "한다", 2: "할 것이다"}             # 시제별 한국어 끝
print("동작", len(VERB), "· 시간 표현", sum(len(v) for v in ADV.values()))  # 개수 확인

### 1-3. 한국어 조사와 영어 서수
한국어는 앞말의 **받침**에 따라 조사가 바뀝니다(작업자**가** · 점검팀**이**, 균열**을** · 누수**를**). 영어는 층 번호를 **서수**(1st, 2nd, 3rd, 4th … 13th)로 씁니다. 두 규칙을 작은 함수로 만듭니다.

> 💡 **새 파이썬 문법 — `ord()` 와 한글 받침**
>
> `ord("가")` 는 글자의 번호(유니코드)입니다. 한글 음절은 `가`(44032)부터 28개씩 묶여 있어서, `(번호 - 44032) % 28` 이 0이 아니면 받침이 있습니다.

In [ ]:
def has_batchim(word):                        # 마지막 글자에 받침이 있는가?
    code = ord(word[-1]) - ord("가")          # 한글 음절 번호 (가 = 0)
    return 0 <= code < 11172 and code % 28 != 0  # 28로 나눈 나머지 ≠ 0 → 받침 있음

def josa(word, with_b, without_b):            # 받침에 맞는 조사 붙이기
    return word + (with_b if has_batchim(word) else without_b)  # 예: 균열+을, 누수+를

def ordinal(n):                               # 1 → 1st, 2 → 2nd, 13 → 13th
    suf = "th" if 10 <= n % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suf}"                        # 숫자 + 꼬리말

print(josa("작업자", "이", "가"), josa("점검팀", "이", "가"), josa("누수", "을", "를"))  # 조사
print(ordinal(1), ordinal(2), ordinal(3), ordinal(11), ordinal(13))  # 서수
check("13 → 13th", ordinal(13) == "13th", True, tol=0)   # ✅ 11 · 12 · 13 은 th

### 1-4. 문장 쌍 만들기 — 짧은 문장 하나, 긴 문장 둘
`place()` 는 "3층 기둥의 균열" 같은 **장소 · 대상** 덩어리를 한국어 · 영어로 함께 만듭니다. `sentence()` 는 문형 셋 중 하나를 고릅니다.

| 문형 | 한국어 | 영어 |
|---|---|---|
| S1 짧은 문장 | 어제 작업자가 3층 기둥의 균열을 보수했다 | the worker repaired the crack in the column on the 3rd floor yesterday |
| L1 동사 둘 | … 보수하고 감리자에게 보고했다 | … and reported it to the supervisor yesterday |
| L2 대상 둘 | … 3층 기둥의 균열과 7층 보의 처짐을 보수했다 | … and the deflection in the beam on the 7th floor yesterday |

In [ ]:
def place(rng):                                # "3층 기둥의 균열" 한 덩어리 (한, 영)
    fl, (mk, me) = rng.randint(1, 15), rng.choice(MEMBER)  # 층 · 부재
    dk = rng.choice(FITS[mk])                  # 그 부재에 맞는 하자
    en = f"the {DEFECT[dk]} in the {me} on the {ordinal(fl)} floor"  # 영어 덩어리
    return f"{fl}층 {mk}의 {dk}", en               # (한국어, 영어)

def sentence(rng):                             # 문장 쌍 하나 (문형, 한국어, 영어)
    t = rng.randrange(3); ak, ae = rng.choice(ADV[t])     # 시제 · 시간 표현
    sk, se = rng.choice(SUBJ); vk, ve = rng.choice(VERB)  # 사람 · 동작
    pk, pe = place(rng); head = f"{ak} {josa(sk, '이', '가')}"  # 앞부분 "어제 작업자가"
    kind = rng.choice(["S1", "S1", "L1", "L2"])           # 짧은 문장 50 %
    if kind == "S1":                           # 짧은 문장
        ko = f"{head} {josa(pk, '을', '를')} {vk}{KO_END[t]}"  # 한국어
        return kind, ko, f"{se} {ve[t]} {pe} {ae}"  # 영어: 사람 → 동작 → 대상 → 시간
    if kind == "L1":                           # 동사 둘: …하고 ○○에게 보고
        s2k, s2e = rng.choice([s for s in SUBJ if s[0] != sk])  # 다른 사람
        ko = f"{head} {josa(pk, '을', '를')} {vk}하고 {s2k}에게 보고{KO_END[t]}"  # 한국어
        return kind, ko, f"{se} {ve[t]} {pe} and {REPORT[t]} it to {s2e} {ae}"  # 영어
    qk, qe = place(rng)                        # 대상 둘: ○○과 ○○을
    ko = f"{head} {josa(pk, '과', '와')} {josa(qk, '을', '를')} {vk}{KO_END[t]}"  # 한국어
    return kind, ko, f"{se} {ve[t]} {pe} and {qe} {ae}"  # 영어: 대상 둘

### 1-5. 8,000쌍 만들고 나누기 ⏸
같은 한국어 문장이 두 번 나오지 않게 8,000쌍을 만든 뒤, **학습 6,000 · 검증 1,000 · 시험 1,000** 으로 나눕니다.
- **학습**: 가중치를 고치는 데 씁니다.
- **검증**: 학습 중에 "처음 보는 문장도 잘하나"를 확인합니다(가중치는 고치지 않음).
- **시험**: 마지막에 한 번만 채점합니다.

In [ ]:
rng = random.Random(0)                         # 씨앗 0: 누구나 같은 데이터
data, seen = [], set()                         # 문장 쌍 목록 · 이미 나온 한국어 문장
while len(data) < 8000:                        # 8,000쌍이 될 때까지
    kind, ko, en = sentence(rng)               # 한 쌍 만들기
    if ko not in seen:                         # ⚠️ 같은 문장은 한 번만
        seen.add(ko); data.append((kind, ko, en))  # 기록하고 담기
random.Random(1).shuffle(data)                 # 순서 섞기 (씨앗 1)
train_data, val_data, test_data = data[:6000], data[6000:7000], data[7000:]  # 6 : 1 : 1
print("학습", len(train_data), "· 검증", len(val_data), "· 시험", len(test_data))  # 개수
print("문형 개수:", dict(Counter(k for k, _, _ in data)))  # S1 · L1 · L2
for kind, ko, en in train_data[:3]:            # 앞의 세 쌍 보기
    print(f"[{kind}] {ko}\n     → {en}")       # 한국어 → 영어
check("데이터 수", len(data), 8000, tol=0)      # ✅ 8,000쌍

🔍 **결과 읽는 법** — 한국어 문장의 끝(`보수했다`)이 영어에서는 두 번째 단어(`repaired`)로 옵니다. 시간 표현(`어제`)은 한국어에서 맨 앞, 영어에서 맨 끝입니다. 문장마다 층 · 부재 · 하자 · 사람이 바뀌므로, 모델은 **어느 한국어 덩어리가 영어의 어느 자리로 가는지**를 배워야 합니다.

---
## 2. 토큰과 텐서  🎬 EP2

### 2-1. 어휘 사전 — 한국어는 글자, 영어는 단어
- **입력(한국어)**: 글자 하나가 토큰입니다. 띄어쓰기도 하나의 토큰입니다. 조사가 붙어도 글자는 몇 종류뿐이라 사전이 작습니다.
- **출력(영어)**: 띄어쓰기로 자른 **단어**가 토큰입니다.
- **특수 토큰** 셋을 맨 앞에 둡니다: `<pad>`(길이 맞춤 빈칸, 0) · `<sos>`(디코더 시작 신호, 1) · `<eos>`(문장 끝, 2)

In [ ]:
SPECIALS = ["<pad>", "<sos>", "<eos>"]         # 특수 토큰 3개 → 번호 0, 1, 2
PAD, SOS, EOS = 0, 1, 2                        # 번호를 이름으로 기억
src_chars = sorted({ch for _, ko, _ in data for ch in ko})  # 한국어에 나온 글자 전부
src_vocab = SPECIALS + src_chars               # 입력 어휘: 특수 토큰 + 글자
words = {w for _, _, en in data for w in __________}  # ✏️ 영어 단어 전부 (힌트: 띄어쓰기로 자르기)
tgt_vocab = SPECIALS + sorted(words)            # 출력 어휘: 특수 토큰 + 단어
src_index = {tok: i for i, tok in enumerate(src_vocab)}  # 글자 → 번호
tgt_index = {tok: i for i, tok in enumerate(tgt_vocab)}  # 단어 → 번호
print("입력 어휘:", len(src_vocab), "· 출력 어휘:", len(tgt_vocab))  # 📐 사전 크기
print("출력 단어 예:", tgt_vocab[3:11])         # 특수 토큰 다음 몇 개
ids3 = [src_index["<pad>"], tgt_index["<sos>"], tgt_index["<eos>"]]  # 특수 토큰 번호 3개
check("특수 토큰 번호", ids3, [0, 1, 2], tol=0)  # ✅ 0, 1, 2

### 2-2. 번호 텐서 만들기 — 패딩과 teacher forcing
문장마다 길이가 다르므로, 미니배치에서 **가장 긴 문장에 맞춰 `<pad>`(0)로 채운** 직사각형 텐서를 만듭니다. 디코더용으로는 텐서를 **두 개** 만듭니다.

| 이름 | 내용 | 역할 |
|---|---|---|
| `dec_in` | `<sos>` + 정답 단어들 | 디코더 **입력** (정답 앞부분) |
| `dec_out` | 정답 단어들 + `<eos>` | 디코더가 맞혀야 할 **다음 단어** |

같은 정답이 **한 칸 어긋나** 있습니다. 학습할 때 디코더에 자기가 낸 단어 대신 **정답 앞부분**을 넣어 주는 방법을 **teacher forcing** 이라고 합니다(💡 이론 시간 「Seq2Seq 학습」).

In [ ]:
def make_batch(pairs):                         # (문형, 한국어, 영어) 묶음 → 번호 텐서 3개
    S = max(len(ko) for _, ko, _ in pairs)     # 가장 긴 한국어 (글자 수)
    T = max(len(en.split()) for _, _, en in pairs) + 1  # 가장 긴 영어 + 1 (<sos>/<eos> 자리)
    src = torch.full((len(pairs), S), PAD)     # 전부 <pad>(0)로 채운 판
    dec_in = torch.full((len(pairs), T), PAD)  # 디코더 입력 판
    dec_out = torch.full((len(pairs), T), PAD) # 디코더 정답 판
    for k, (_, ko, en) in enumerate(pairs):    # 한 쌍씩 채우기
        src[k, :len(ko)] = torch.tensor([src_index[ch] for ch in ko])  # 글자 → 번호
        ids = [tgt_index[w] for w in en.split()]   # 단어 → 번호
        dec_in[k, :len(ids) + 1] = torch.tensor(___________)  # ✏️ <sos>+정답 (힌트: 시작 신호 먼저)
        dec_out[k, :len(ids) + 1] = torch.tensor(___________)  # ✏️ 정답+<eos> (힌트: 끝 신호 뒤에)
    return src.to(device), dec_in.to(device), dec_out.to(device)  # 계산 장치로

In [ ]:
src, dec_in, dec_out = make_batch(train_data[:3])  # 첫 세 쌍으로 시험
print("📐 src", tuple(src.shape), "· dec_in", tuple(dec_in.shape))  # [배치, 길이]
print("📐 dec_out", tuple(dec_out.shape))     # dec_in 과 같은 모양
print("한국어 첫 문장 번호:", src[0, :12].tolist(), "…")    # 글자 번호 (앞 12개)
n = len(train_data[0][2].split()) + 1          # 첫 문장의 영어 단어 수 + 1
print("dec_in :", [tgt_vocab[i] for i in dec_in[0, :6].tolist()], "…")   # <sos> 로 시작
print("dec_out:", [tgt_vocab[i] for i in dec_out[0, :6].tolist()], "…")  # 한 칸 앞선 단어
print("dec_out 끝:", [tgt_vocab[i] for i in dec_out[0, n - 3:n].tolist()])  # <eos> 로 끝
check("한 칸 어긋남", dec_in[0, 1:n], dec_out[0, :n - 1], tol=0)  # ✅ 같은 정답

🔍 **결과 읽는 법** — `dec_in` 의 둘째 칸부터가 `dec_out` 과 똑같습니다. 디코더는 **`<sos>` 를 보고 `the`, `the` 를 보고 `site` …** 처럼 매 칸마다 **다음 단어를 고르는 분류**를 합니다. 짧은 문장의 뒤쪽은 `<pad>`(0)로 채워져 있습니다.

### 2-3. 임베딩 — 번호를 벡터로
번호 `5` 와 `6` 은 크기 비교에 아무 뜻이 없습니다. 그래서 번호마다 **학습할 수 있는 벡터**를 하나씩 붙입니다. 이것이 **임베딩**입니다. `padding_idx=PAD` 를 주면 `<pad>` 의 벡터는 0으로 두고 학습하지 않습니다.

In [ ]:
torch.manual_seed(0)                           # 같은 출발점
emb = nn.Embedding(len(src_vocab), 32, padding_idx=PAD)  # 글자 번호 → 32칸 벡터
emb = emb.to(device)                           # 계산 장치로 옮기기 (GPU 런타임이면 꼭 필요)
vecs = emb(src)                                # 번호 텐서 → 벡터 텐서
print("📐", tuple(src.shape), "→", tuple(vecs.shape))   # [배치, 길이] → [배치, 길이, 32]
print("<pad> 벡터 합:", emb.weight[PAD].abs().sum().item())  # 0 (학습 안 함)
check("<pad> 벡터는 0", emb.weight[PAD].abs().sum(), 0.0, tol=0)  # ✅

---
## 3. Seq2Seq(GRU) 모델과 학습 루프  🎬 EP3

### 3-1. 인코더와 디코더
- **인코더**: 한국어 글자를 처음부터 끝까지 GRU로 읽고, **마지막 은닉 상태** 하나에 문장 전체를 담습니다. 이것이 **컨텍스트 벡터**입니다.
- **디코더**: 컨텍스트 벡터에서 출발해, 매 칸마다 **다음 영어 단어의 점수**를 냅니다. 디코더에는 `forward` 를 따로 두지 않고, 3-2의 `Seq2Seq` 가 두 부품을 이어서 씁니다.

In [ ]:
class Encoder(nn.Module):                      # 인코더: 한국어 → 은닉 상태들
    def __init__(self, vocab, emb=32, hidden=64):  # 어휘 수 · 임베딩 32 · 은닉 64
        super().__init__()                     # nn.Module 준비
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)  # 글자 번호 → 32칸 벡터
        self.gru = nn.GRU(emb, hidden, batch_first=True)      # 지난 실습의 GRU
    def forward(self, src):                    # src: [배치, 한국어 길이]
        outs, h = self.gru(self.emb(src))      # 모든 시점 출력 · 마지막 은닉 상태
        return outs, h                         # 📐 [B, S, 64] · [1, B, 64]

class Decoder(nn.Module):                      # 디코더: 앞 단어들 → 다음 단어 점수
    def __init__(self, vocab, emb=32, hidden=64):  # 어휘 수 · 임베딩 32 · 은닉 64
        super().__init__()                     # nn.Module 준비
        self.emb = nn.Embedding(vocab, emb, padding_idx=PAD)  # 영어 단어 번호 → 벡터
        self.gru = nn.GRU(emb, hidden, batch_first=True)      # 디코더용 GRU (가중치 따로)
        self.out = nn.Linear(hidden, vocab)    # 64 → 단어별 점수

### 3-2. 두 부품 잇기 — 컨텍스트 벡터가 다리
인코더의 마지막 은닉 상태 `h` 를 디코더 GRU의 **첫 은닉 상태**로 넣습니다. 한국어 문장의 정보는 **오직 이 64칸 벡터 하나**로만 디코더에 전달됩니다.

In [ ]:
class Seq2Seq(nn.Module):                      # 인코더 + 디코더
    def __init__(self, hidden=64):             # 은닉 64
        super().__init__()                     # nn.Module 준비
        self.encoder = Encoder(len(src_vocab), hidden=hidden)  # 한국어용 인코더
        self.decoder = Decoder(len(tgt_vocab), hidden=hidden)  # 영어용 디코더
    def forward(self, src, dec_in):            # 한국어 번호 · 디코더 입력(<sos> + 정답)
        _, h = self.encoder(src)               # 컨텍스트 벡터 = 마지막 은닉 상태
        outs, _ = self.decoder.gru(self.decoder.emb(dec_in), h)  # h 에서 출발
        return ________________(outs)          # ✏️ 📐 [B, T, V] 점수 (힌트: 출력층)

torch.manual_seed(0)                           # 같은 출발점 (재현용)
model = Seq2Seq().to(device)                   # 모델 만들기
logits = model(src, dec_in)                    # 세 쌍으로 순전파 시험
print("📐 logits", tuple(logits.shape))        # [3, 영어 길이 + 1, 출력 어휘 수]
print("파라미터 수:", sum(p.numel() for p in model.parameters()))  # 학습할 숫자 개수

### 3-3. 손실 — 칸마다 다음 단어 분류
칸마다 "다음 단어가 무엇인가"를 맞히는 **분류 문제**이므로, 지난 실습의 **교차 엔트로피**를 그대로 씁니다. 다만 `<pad>` 칸은 정답이 없으니 `ignore_index=PAD` 로 **채점에서 뺍니다.**
점수 텐서 `[B, T, V]` 와 정답 `[B, T]` 를 각각 `[B·T, V]` · `[B·T]` 로 펴서 한꺼번에 채점합니다.

In [ ]:
loss_fn = nn.CrossEntropyLoss(________________)  # ✏️ <pad> 채점 제외 (힌트: 패드 칸은 빼기)
V = len(tgt_vocab)                             # 출력 어휘 수 (= 분류할 종류 수)
loss = loss_fn(logits.reshape(-1, V), dec_out.reshape(-1))  # [B·T, V] vs [B·T]
print(f"학습 전 손실: {loss.item():.4f}")        # 아무것도 모르는 상태
print(f"찍기 수준 ln(V) = {math.log(V):.4f}")   # 💡 V개 중 무작위 추측
lnV = math.log(V)                              # 찍기 수준
check_range("학습 전 손실 ≈ ln(V)", loss.item(), lnV - 0.5, lnV + 0.5)  # ✅

### 3-4. 학습 루프 — 지난 실습과 같은 4단계
같은 함수 `run_epoch()` 로 **학습**(가중치 고침)과 **검증**(가중치 안 고침)을 모두 합니다. 옵티마이저를 주면 학습, 안 주면 검증입니다. 5–7장의 어텐션 · Transformer 도 이 함수를 그대로 씁니다.

| 단계 | 코드 |
|:-:|---|
| 1️⃣ 순전파 | `logits = model(src, dec_in)` — teacher forcing |
| 2️⃣ 오차 | `loss_fn(logits 펴기, dec_out 펴기)` — 다음 단어 분류 |
| 3️⃣ 역전파 | `zero_grad()` → `backward()` (+ 기울기 자르기) |
| 4️⃣ 업데이트 | `optimizer.step()` |

In [ ]:
def run_epoch(model, pairs, optimizer=None, seed=0):  # 한 바퀴 (옵티마이저 있으면 학습)
    model.train(optimizer is not None)         # 학습 모드 / 평가 모드
    order = list(range(len(pairs)))            # 문장 번호
    if optimizer: random.Random(seed).shuffle(order)  # 학습 때만 섞기 (씨앗 고정)
    total = 0.0                                # 손실 합계
    with torch.set_grad_enabled(optimizer is not None):  # 검증 때는 미분 끄기
        for b in range(0, len(order), 64):     # 64쌍씩
            src, dec_in, dec_out = make_batch([pairs[i] for i in order[b:b + 64]])
            logits = model(src, dec_in)        # 1️⃣ 순전파 (teacher forcing)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))  # 2️⃣
            if optimizer:                      # 학습일 때만
                optimizer.zero_grad(); loss.________()  # ✏️ 3️⃣ 역전파 (힌트: 기울기 계산)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 기울기 폭주 막기
                optimizer.____()               # ✏️ 4️⃣ 업데이트 (힌트: 가중치 고치기)
            total += loss.item() * len(src)    # 평균 × 개수
    return total / len(pairs)                  # 평균 손실

### 3-5. Seq2Seq 학습 — 25 에포크 ⏸
`fit()` 은 에포크마다 학습 손실과 **검증 손실**을 기록해 두고, 5 에포크마다 출력합니다. 은닉 64 · 임베딩 32 · Adam(학습률 0.001) · 미니배치 64 입니다.

⏸ 이 노트북을 만든 컴퓨터(CPU)에서 약 1분 걸립니다. GPU 런타임이면 더 빠릅니다.

In [ ]:
def fit(model, epochs, lr=1e-3):               # 학습 + 매 에포크 검증 손실 기록
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # Adam
    history, start = [], time.time()           # (학습, 검증) 손실 기록 · ⏱ 시작
    for epoch in range(1, epochs + 1):         # 에포크 반복
        tr = run_epoch(model, train_data, optimizer, seed=epoch)  # 학습 한 바퀴
        va = run_epoch(model, val_data)        # 검증 (가중치 그대로)
        history.append((tr, va))               # 기록
        if epoch % 5 == 0:                     # 5 에포크마다 출력
            print(f"에포크 {epoch:2d}: 학습 손실 {tr:.4f} · 검증 손실 {va:.4f}")
    sec = time.time() - start                  # ⏱ 걸린 시간
    print(f"⏱ {sec:.0f}초")                    # 학습 시간
    return history, sec                        # 곡선 · 비교표에 씀

torch.manual_seed(0)                           # 같은 출발점 (재현용)
s2s = Seq2Seq().to(device)                     # 새 Seq2Seq
s2s_hist, s2s_sec = fit(s2s, epochs=25)        # 25 에포크 학습

### 3-6. loss 곡선 읽기
가로축은 에포크, 세로축은 손실입니다. 두 곡선을 함께 봅니다.
- **둘 다 내려가면** 잘 배우고 있습니다.
- **학습 손실만 내려가고 검증 손실이 다시 오르면** 학습 문장을 외우기 시작한 것(과적합)입니다.

In [ ]:
def plot_curves(curves):                       # {이름: 기록} → 손실 곡선
    plt.figure(figsize=(7, 3.6))               # 그림 크기
    for name, hist in curves.items():          # 모델마다
        ep = range(1, len(hist) + 1)           # 가로축: 에포크
        plt.plot(ep, [t for t, _ in hist], "--", label=f"{name} train")  # 학습 (점선)
        plt.plot(ep, [v for _, v in hist], "-", label=f"{name} val")     # 검증 (실선)
    plt.xlabel("epoch"); plt.ylabel("loss")     # 축 이름 (영어: Colab 한글 글꼴 없음)
    plt.legend(); plt.grid(alpha=0.3); plt.show()  # 범례 · 격자 · 그리기

plot_curves({"Seq2Seq": s2s_hist})             # Seq2Seq 곡선
drop = s2s_hist[0][1] - s2s_hist[-1][1]       # 검증 손실이 줄어든 양
check_range("검증 손실 감소 (처음 → 끝)", drop, 0.3, 5.0)  # ✅

🔍 **결과 읽는 법** — 두 곡선이 함께 내려가다가 점점 평평해집니다. 검증 손실이 다시 오르지 않으니 외우기(과적합)는 아닙니다. 하지만 손실이 0에 가깝게 내려가지 않고 멈춥니다. 무엇을 틀리는지는 다음 장에서 직접 번역해 보며 확인합니다.

---
## 4. 번역해 보기와 채점  🎬 EP4

### 4-1. 탐욕 디코딩 — 시험 때는 정답이 없다
학습 때는 디코더에 **정답 앞부분**(teacher forcing)을 넣었지만, 시험 때는 정답이 없습니다. 그래서 `<sos>` 하나로 시작해 **점수가 가장 큰 단어**(argmax)를 고르고, 그 단어를 입력 끝에 붙여 다시 다음 단어를 고릅니다. 매번 1등만 고르므로 **탐욕(greedy) 디코딩**이라 부릅니다. `<eos>` 가 나오면 끝입니다.

In [ ]:
@torch.no_grad()                               # 채점만 (미분 추적 끄기)
def greedy_decode(model, src, max_len=30):     # 한 단어씩 이어 붙이기
    model.eval()                               # 평가 모드
    seq = torch.full((src.size(0), 1), SOS, device=src.device)  # 모두 <sos> 로 시작
    for _ in range(max_len):                   # 최대 30 단어
        logits = model(src, seq)               # 지금까지 낸 단어를 입력으로
        nxt = logits[:, -1].______(-1, keepdim=True)  # ✏️ 마지막 칸 1등 (힌트: 최댓값 번호)
        seq = _________([seq, nxt], dim=1)     # ✏️ 끝에 붙이기 (힌트: 텐서 이어 붙이기)
    return seq[:, 1:]                          # <sos> 떼고 돌려주기

def ids_to_words(row):                         # 번호 한 줄 → 단어 목록 (<eos> 에서 멈춤)
    out = []                                   # 결과 목록
    for i in row.tolist():                     # 번호마다
        if i == EOS: break                     # <eos> 면 끝
        out.append(tgt_vocab[i])               # 번호 → 단어
    return out                                 # 단어 목록

In [ ]:
def translate(model, ko):                      # 한국어 문장 하나 → 영어 문장
    src, _, _ = make_batch([("", ko, "")])     # 한 문장짜리 미니배치
    return " ".join(ids_to_words(greedy_decode(model, src)[0]))  # 단어 → 문장

for kind, ko, en in test_data[:3]:             # 시험 문장 세 개
    print(f"[{kind}] {ko}")                    # 한국어
    print(f"  정답 → {en}")                    # 정답 영어
    print(f"  모델 → {translate(s2s, ko)}")     # Seq2Seq 번역

🔍 **결과 읽는 법** — 문장의 **뼈대**(사람 → 동작 → the … in the … on the … floor → 시간)는 잘 따라 합니다. 그런데 **층 번호 · 부재 · 하자** 같은 세부 내용을 자주 바꿔 씁니다. 긴 문장 전체를 64칸 벡터 하나에 담다 보니 세부가 흐려지는 것입니다.

### 4-2. 번역 점수 BLEU — 단어 묶음이 얼마나 겹치나
**BLEU**(0–100)는 모델 번역의 **1–4단어 묶음**이 정답에 얼마나 들어 있는지(정밀도)를 재서 기하평균을 내고, 번역이 정답보다 짧으면 **짧음 벌점**을 곱합니다. 여기서는 시험 문장 전체를 모아 한 번에 계산합니다(코퍼스 BLEU).

In [ ]:
def bleu(hyps, refs):                          # 단어 목록들 → 코퍼스 BLEU (0–100)
    hit, tot, h_len, r_len = [0] * 4, [0] * 4, 0, 0  # n별 맞힌 묶음 · 전체 묶음 · 길이
    for h, r in zip(hyps, refs):               # 문장마다
        h_len += len(h); r_len += len(r)       # 길이 누적
        for n in range(1, 5):                  # 1 ~ 4단어 묶음
            hc = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))  # 번역 묶음
            rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))  # 정답 묶음
            hit[n - 1] += sum(min(c, rc[g]) for g, c in hc.items())  # 정답 횟수까지만 인정
            tot[n - 1] += max(len(h) - n + 1, 0)   # 번역의 n단어 묶음 수
    if min(hit) == 0: return 0.0               # 하나라도 0이면 점수 0
    bp = 1.0 if h_len > r_len else math.exp(1 - r_len / h_len)  # 짧음 벌점
    logp = sum(math.log(hit[i] / tot[i]) for i in range(4)) / 4  # 정밀도 로그 평균
    return 100 * bp * math.exp(logp)           # 기하평균 × 벌점 × 100

ref = "the worker repaired the crack in the column on the 3rd floor yesterday".split()  # 정답
print(f"같은 문장: {bleu([ref], [ref]):.1f}")       # 100
check("똑같으면 100", bleu([ref], [ref]), 100.0, tol=1e-6)  # ✅

### 4-3. 시험 문장 1,000개 채점 ⏸
`evaluate()` 는 시험 문장을 250개씩 번역해 **BLEU** 와 **완전 일치**(영어 문장 전체가 정답과 똑같은 비율)를 계산하고, 문형(짧은 S1 · 긴 L1 · L2)별로도 나눠 봅니다. 5–7장에서 어텐션 · Transformer 도 같은 함수로 채점합니다.

In [ ]:
def evaluate(model, pairs):                    # 번역 → BLEU · 완전 일치 (전체 · 문형별)
    hyps = []                                  # 모델 번역 (단어 목록)
    for b in range(0, len(pairs), 250):        # 250쌍씩
        src, _, _ = make_batch(pairs[b:b + 250])   # 한국어만 쓰임
        hyps += [ids_to_words(row) for row in greedy_decode(model, src)]  # 번역
    refs = [en.split() for _, _, en in pairs]  # 정답 (단어 목록)
    em = sum(h == r for h, r in zip(hyps, refs)) / len(refs)  # 완전 일치 비율
    res = {"all": (bleu(hyps, refs), em)}       # 전체 점수
    for kind in ("S1", "L1", "L2"):            # 문형별
        ii = [i for i, p in enumerate(pairs) if p[0] == kind]  # 그 문형의 번호
        res[kind] = (bleu([hyps[i] for i in ii], [refs[i] for i in ii]),
                     sum(hyps[i] == refs[i] for i in ii) / len(ii))  # (BLEU, 완전 일치)
    return res, hyps                           # 점수 · 번역

s2s_res, s2s_hyps = evaluate(s2s, test_data)   # Seq2Seq 채점
for k, (b, em) in s2s_res.items():             # 전체 · 문형별
    print(f"{k:>3}: BLEU {b:5.1f} · 완전 일치 {em:.3f}")  # 점수 표
check_range("Seq2Seq 시험 BLEU", s2s_res["all"][0], 25, 60)  # ✅ 뼈대는 맞힘

🔍 **결과 읽는 법** — BLEU는 40점 안팎이지만 **완전 일치는 거의 0** 입니다. 단어 묶음은 꽤 겹치는데, 문장마다 어딘가 한 곳은 틀린다는 뜻입니다. 특히 대상이 둘인 긴 문장(L2)에서 점수가 낮습니다.

이것이 **컨텍스트 벡터 하나의 병목**입니다. 다음 장에서는 디코더가 단어를 낼 때마다 **한국어의 필요한 글자를 다시 보게** 하는 **어텐션**을 붙여, 같은 데이터 · 같은 학습 루프로 다시 학습합니다.

---
## 다음 이어서 할 것
- **5장 어텐션** — 디코더가 매 단어마다 인코더의 모든 시점을 다시 보는 Seq2Seq + Attention
- **6장 Transformer** — 셀프 어텐션 · 위치 임베딩 · 마스크
- **7장 비교와 오류 분석** — 세 모델을 같은 표로 비교하고, 다른 전문 분야로 바꿔 적용하기

5–7장은 다음 공개분에서 이 노트북에 이어서 추가됩니다.